# Professional Quant Backtesting

In [ ]:
def full_backtest(signal_proba, actual_returns, threshold=0.5,
                  transaction_cost=0.001, name="Strategy"):
    """
    Full backtest engine with transaction costs and risk metrics.

    signal_proba:     model's predicted probability of an "up" day
    actual_returns:   the actual daily returns that occurred
    threshold:        probability above which we go long
    transaction_cost: 0.1% per trade (0.001) — realistic for retail
    """
    signals   = (signal_proba > threshold).astype(int)
    prev_sig  = signals.shift(1).fillna(0)
    trades    = (signals != prev_sig).astype(int)  # 1 on days we change position

    # Strategy return = signal * return - transaction cost on trade days
    strategy_returns = signals * actual_returns - trades * transaction_cost

    # Core performance metrics
    total_days  = len(strategy_returns)
    trading_days_per_year = 252

    # Cumulative return
    cum_return_strat  = (1 + strategy_returns).prod() - 1
    cum_return_bh     = (1 + actual_returns).prod() - 1

    # Annualized return
    years = total_days / trading_days_per_year
    ann_return = (1 + cum_return_strat) ** (1/years) - 1

    # Annualized volatility
    ann_vol = strategy_returns.std() * np.sqrt(trading_days_per_year)

    # Sharpe Ratio (risk-free rate = 5% annual = 0.0198% daily)
    rf_daily = 0.05 / trading_days_per_year
    excess   = strategy_returns - rf_daily
    sharpe   = (excess.mean() / excess.std()) * np.sqrt(trading_days_per_year)

    # Sortino Ratio: like Sharpe but only penalizes DOWNSIDE volatility
    downside_returns = strategy_returns[strategy_returns < rf_daily]
    downside_vol     = downside_returns.std() * np.sqrt(trading_days_per_year)
    sortino          = (ann_return - 0.05) / downside_vol if downside_vol > 0 else np.nan

    # Maximum Drawdown: worst peak-to-trough decline
    cum_equity = (1 + strategy_returns).cumprod()
    rolling_max = cum_equity.cummax()
    drawdown    = (cum_equity - rolling_max) / rolling_max
    max_dd      = drawdown.min()

    # Calmar Ratio: annualized return / max drawdown (higher is better)
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    # Win rate and profit factor
    wins   = strategy_returns[strategy_returns > 0]
    losses = strategy_returns[strategy_returns < 0]
    win_rate      = len(wins) / len(strategy_returns[strategy_returns != 0])
    profit_factor = wins.sum() / abs(losses.sum()) if len(losses) > 0 else np.nan

    # Trade count and average holding period
    n_trades     = trades.sum()
    days_in_mkt  = signals.sum()

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Period:              {total_days} days ({years:.1f} years)")
    print(f"  Trades:              {n_trades}")
    print(f"  Days in market:      {days_in_mkt} ({days_in_mkt/total_days:.1%})")
    print(f"  ─────────────────────────────────────────")
    print(f"  Cumulative return:   {cum_return_strat:.2%}")
    print(f"  Annualized return:   {ann_return:.2%}")
    print(f"  Annualized vol:      {ann_vol:.2%}")
    print(f"  ─────────────────────────────────────────")
    print(f"  Sharpe Ratio:        {sharpe:.3f}")
    print(f"  Sortino Ratio:       {sortino:.3f}")
    print(f"  Calmar Ratio:        {calmar:.3f}")
    print(f"  Max Drawdown:        {max_dd:.2%}")
    print(f"  ─────────────────────────────────────────")
    print(f"  Win rate:            {win_rate:.2%}")
    print(f"  Profit factor:       {profit_factor:.3f}")
    print(f"{'='*50}")

    return {
        'cum_returns':    cum_equity,
        'drawdown':       drawdown,
        'strategy_rets':  strategy_returns,
        'sharpe':         sharpe,
        'max_dd':         max_dd,
        'ann_return':     ann_return
    }


# Run the backtest on our holdout set
actual_rets_holdout = df_features.loc[X_holdout.index, 'return_1d']

backtest_result = full_backtest(
    signal_proba      = pd.Series(tuned_proba, index=X_holdout.index),
    actual_returns    = actual_rets_holdout,
    threshold         = 0.5,
    transaction_cost  = 0.001,
    name              = "XGBoost ML Strategy"
)

# Buy-and-hold benchmark
bh_result = full_backtest(
    signal_proba      = pd.Series(np.ones(len(X_holdout)), index=X_holdout.index),
    actual_returns    = actual_rets_holdout,
    threshold         = 0.5,
    transaction_cost  = 0,
    name              = "Buy and Hold (AAPL)"
)

In [ ]:
# Threshold optimization
# 0.5 is not necessarily the best threshold.
# A higher threshold = fewer but higher-confidence trades

thresholds = np.arange(0.4, 0.65, 0.02)
threshold_results = []

for t in thresholds:
    signals       = (pd.Series(tuned_proba) > t).astype(int)
    strat_returns = signals.values * actual_rets_holdout.values - \
                    (signals.values != np.roll(signals.values, 1)).astype(int) * 0.001
    strat_returns = pd.Series(strat_returns)
    rf            = 0.05 / 252
    sharpe        = ((strat_returns - rf).mean() / strat_returns.std()) * np.sqrt(252)
    cum_ret       = (1 + strat_returns).prod() - 1
    threshold_results.append({'threshold': t, 'sharpe': sharpe, 'cum_return': cum_ret,
                               'n_trades': (signals != signals.shift(1)).sum(),
                               'pct_long': signals.mean()})

thresh_df = pd.DataFrame(threshold_results)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(thresh_df['threshold'], thresh_df['sharpe'], 'o-', color='steelblue')
axes[0].set_title('Sharpe Ratio vs Threshold')
axes[0].set_xlabel('Threshold'); axes[0].grid(True, alpha=0.3)

axes[1].plot(thresh_df['threshold'], thresh_df['cum_return'], 'o-', color='green')
axes[1].set_title('Cumulative Return vs Threshold')
axes[1].set_xlabel('Threshold'); axes[1].grid(True, alpha=0.3)

axes[2].plot(thresh_df['threshold'], thresh_df['n_trades'], 'o-', color='orange')
axes[2].set_title('Number of Trades vs Threshold')
axes[2].set_xlabel('Threshold'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_thresh = thresh_df.loc[thresh_df['sharpe'].idxmax(), 'threshold']
print(f"\nOptimal threshold: {best_thresh:.2f}")
print(thresh_df.set_index('threshold').round(3))

In [ ]:
# Equity curve with drawdown visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(backtest_result['cum_returns'].index,
         backtest_result['cum_returns'].values,    label='ML Strategy', color='steelblue', lw=2)
ax1.plot(bh_result['cum_returns'].index,
         bh_result['cum_returns'].values,          label='Buy & Hold',  color='gray', lw=1.5, ls='--')
ax1.set_title('Equity Curve: ML Strategy vs Buy and Hold')
ax1.set_ylabel('Portfolio Value (start = 1.0)')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.fill_between(backtest_result['drawdown'].index,
                 backtest_result['drawdown'].values, 0, color='red', alpha=0.4)
ax2.set_title('Strategy Drawdown')
ax2.set_ylabel('Drawdown')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()